# PC40 — Lab 05: Model Validation and Parallel Cross-Validation

**Language:** Python / scikit-learn · **Platform:** local multicore CPU · **Prerequisites:** PC40 Labs 02–04.

The subject is parallel programming applied to model validation. The aim is not the most accurate predictor, but to implement, validate and benchmark the computational structure of nested cross-validation.

| Part | Activity |
|:---|:---|
| 1 | Hold-out, k-fold, LOOCV, nested CV |
| 2 | Sequential nested CV; dependencies and reductions |
| 3 | Parallelise folds / hyperparameters; explicit TLP |
| 4 | Benchmark $p = 1, 2, 4, 8, 16$; speedup and efficiency |
| 5 | Decision tree vs gradient boosting vs XGBoost |

`GridSearchCV` and `n_jobs=-1` are not used anywhere in the required implementation: every loop is written out so that the computational structure is visible before it is parallelised.

The implementation lives in `nested_cv.py` beside this notebook, so the notebook and the benchmark scripts cannot drift apart. Each function is printed here where it is first used.

## 0 — Environment

The explicit task parallelism is the thing being measured, so the native thread pools of numpy and scikit-learn are pinned to one thread. Those variables are only read when the library initialises, so `nested_cv` sets them at the top of the module, before it imports numpy.

In [1]:
import inspect, os, platform, statistics, time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

import nested_cv as ncv          # pins OMP_NUM_THREADS etc. on import
import numpy as np
import sklearn

print('Python      :', platform.python_version())
print('scikit-learn:', sklearn.__version__, '| numpy:', np.__version__)
print('Platform    :', platform.platform())
print('Logical CPUs:', os.cpu_count(), '| affinity:', len(os.sched_getaffinity(0)))
print('Load average:', [round(v, 2) for v in os.getloadavg()])
print('Native thread limits:', ncv.native_thread_settings())


def timed(fn):
    '''Wall time, and the ratio of process CPU time to wall time.

    The second number says whether the workers really ran: p threads executing
    concurrently consume close to p CPU seconds per wall second, while threads
    taking turns on the GIL stay near 1 whatever p is.
    '''
    c0, w0 = time.process_time(), time.perf_counter()
    out = fn()
    wall = time.perf_counter() - w0
    return out, wall, (time.process_time() - c0) / wall


def best_of(fn, n=3):
    fn()
    runs = [timed(fn)[1:] for _ in range(n)]
    best = min(runs, key=lambda r: r[0])
    return best[0], best[1]

Python      : 3.14.4
scikit-learn: 1.9.1 | numpy: 2.5.5
Platform    : Linux-7.0.0-43-generic-x86_64-with-glibc2.43
Logical CPUs: 16 | affinity: 16
Load average: [3.27, 3.6, 3.5]
Native thread limits: {'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1', 'VECLIB_MAXIMUM_THREADS': '1'}


## 1 — Why model validation matters

**Training, validation and test are three different roles.** Training data estimate the *parameters* of the model. Validation data compare candidate *hyperparameters*. Test data are used once, to estimate how the chosen procedure generalises. Using the same observations for the last two roles reports the score of the configuration that happened to suit those observations, which is optimistic rather than wrong-by-accident.

**The progression.** A single hold-out split is cheap but unstable: on 25 observations, a 20% test set is five points and the measured error depends heavily on which five. *k*-fold cross-validation divides the data into $k$ folds and uses each once as the held-out fold, so every observation is predicted exactly once by a model that did not see it. **LOOCV** is the extreme case $k = N$: 24 observations train, 1 is predicted, repeated 25 times.

**Why nest.** If hyperparameters are also chosen by cross-validation, a single loop is not enough: the fold used to pick `max_depth` has already influenced the model, so scoring on it is no longer independent. Nested CV separates the two jobs — the **outer** loop (LOOCV here) estimates generalisation, and the **inner** loop (5-fold here) selects hyperparameters inside each outer training set. The outer test observation takes part in neither the fitting nor the selection.

### Checkpoint — Synthesis 1/4

Training fits parameters, validation ranks hyperparameters, test estimates generalisation. The inner loop does model selection on the 24 outer-training observations; the outer loop measures how well that whole selection procedure generalises. The held-out observation must stay invisible during selection because any configuration chosen with knowledge of it is chosen partly to fit it, and the reported error would then measure fit, not generalisation.

## 2 — The problem instance

The brief states the problem for 25 observations but supplies no dataset, so one is generated from a written-down function with a fixed seed. The response is smooth rather than piecewise constant, so a regression tree can only approximate it: a deeper tree fits the training data better while generalising worse, which is what makes the validation protocol worth running.

In [2]:
print(inspect.getsource(ncv.make_dataset))
X, y = ncv.make_dataset()
grid = ncv.build_grid()
print('X', X.shape, ' y', y.shape, f' y in [{y.min():.2f}, {y.max():.2f}]')
print('hyperparameter grid:', ncv.PARAM_GRID)
print('configurations H =', len(grid))

outer_train = np.array([j for j in range(25) if j != 0])
print('inner fold sizes for one outer iteration:',
      [len(va) for _, va in ncv.inner_splits(outer_train)])
print('model fits in one full nested CV:', 25 * (len(grid) * 5 + 1))

def make_dataset(n_samples: int = 25, seed: int = SEED):
    """The lab states the problem for 25 observations but supplies no data, so
    the dataset is generated here and the generating function is written down,
    which makes the whole experiment reproducible from this file alone.

        x1, x2, x3 ~ U(0, 1)
        y = 8*x1 + 5*sin(3*x2) - 4*x3^2 + e,     e ~ N(0, 0.5^2)

    The response is deliberately not a step function: a regression tree can only
    approximate it, so a deeper tree fits the training data better while
    generalizing worse.  That is what makes the validation protocol meaningful
    rather than decorative.
    """
    rng = np.random.default_rng(seed)
    X = rng.uniform(0.0, 1.0, size=(n_samples, 3))
    noise = rng.normal(0.0, 0.5, size=n_samples)
    y = 8.0 * X[:, 0] + 5.0 * np.sin(3.0 * X[:, 1]) - 4.0 * X[:, 2] ** 2 + noise
    return X, y

X (25, 3)  y (25,)  y in [0.54, 10.93]
hyperparameter grid: {'max_depth': [2, 3, 4, 5], 'min_samples_split': [2,

### Determinism

The answer must not depend on how the work is scheduled, so nothing in the algorithm is allowed to vary with execution order:

* the grid is built once, in a fixed order, and tasks are indexed into it;
* `KFold(shuffle=False)` gives the 5+5+5+5+4 inner split of the brief and never draws a random number;
* every estimator is built with `random_state=42`;
* the minimum over configurations is taken **after** the tasks finish, in grid order, with a strict `<` so the first minimum wins.

A parallel run must therefore reproduce the sequential external predictions *exactly*, not merely closely.

## 3 — Sequential nested CV

One outer iteration holds out observation $i$, splits the other 24 into 5 inner folds, scores all 36 configurations over those folds, reduces to the best one, refits it on all 24 and predicts the held-out observation.

In [3]:
print(inspect.getsource(ncv.outer_iteration))
print(inspect.getsource(ncv.select_best))

def outer_iteration(i, X, y, grid, kind="tree", inner_pool=None):
    """Hold out observation i, select hyperparameters on the other 24, refit on
    all 24 and predict the held-out one.

    `inner_pool` selects decomposition C: when it is given, the 5 inner folds of
    each configuration are submitted to it instead of being looped over.
    """
    n = X.shape[0]
    outer_train_idx = np.array([j for j in range(n) if j != i])
    splits = inner_splits(outer_train_idx)

    scores = []
    for params in grid:
        if inner_pool is None:
            scores.append(mean_inner_score(X, y, splits, params, kind))
        else:
            task = functools.partial(_fold_task, X=X, y=y, params=params,
                                     kind=kind)
            fold_errors = list(inner_pool.map(task, splits))
            scores.append(float(np.mean(fold_errors)))

    _, best_score, best_params = select_best(scores, grid)

    final = make_model(best_params, kind)
    final.fit(X[outer_tra

In [4]:
seq, t_seq, _ = timed(lambda: ncv.nested_cv_sequential(X, y, grid))
m = ncv.metrics(seq)
print(f'sequential nested CV: {t_seq:.3f} s')
print('MAE  %.4f' % m['MAE'], '| RMSE %.4f' % m['RMSE'], '| R2 %.4f' % m['R2'])
print('\nthe 25 external predictions come from 25 models, none of which saw its own test point:')
for r in seq[:5]:
    print(f"  i={r['index']:2d}  y={r['y_true']:6.3f}  y_hat={r['y_pred']:6.3f}  {r['best_params']}")
print('  ...')

sequential nested CV: 1.193 s
MAE  1.6910 | RMSE 2.1512 | R2 0.3249

the 25 external predictions come from 25 models, none of which saw its own test point:
  i= 0  y= 8.331  y_hat= 8.488  {'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 3}
  i= 1  y= 3.237  y_hat= 7.447  {'max_depth': 2, 'min_samples_split': 2, 'min_samples_leaf': 1}
  i= 2  y= 9.897  y_hat= 7.659  {'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 1}
  i= 3  y= 4.438  y_hat= 7.537  {'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 3}
  i= 4  y= 7.562  y_hat= 7.950  {'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 3}
  ...


### Find the parallelism

1. **Are the 25 outer iterations independent?** Yes. Iteration $i$ reads the shared data, builds its own index sets, and writes only output slot $i$. Nothing it computes is read by another iteration.
2. **Are the 36 configurations independent?** Yes, within one outer iteration. Each is scored on the same 5 inner folds and produces one number. They only meet at the reduction that picks the minimum.
3. **Can the 5 inner folds run in parallel?** Yes. Each fits one model on 4 folds and scores it on the fifth. They meet at the mean.
4. **Where are the reductions?** Two: the mean of the 5 inner errors for one configuration, and the minimum over the 36 configurations. A third, at the very end, aggregates the 25 external predictions into MAE, RMSE and $R^2$.
5. **What can be shared read-only?** `X`, `y` and the hyperparameter grid. They are never written during the computation, and concurrent reads of unchanging data cannot race.
6. **What must be written independently?** One external prediction per outer iteration, one mean score per configuration, one fold error per inner fold — each with exactly one writer.
7. **One thread per model fit?** No. There are 4500 fits and each takes about 0.3 ms, while creating and scheduling a thread costs tens of microseconds and a *process* task costs far more. The number of independent operations is not the number of workers that should execute them.
8. **Which level first?** The outer folds. They give 25 tasks, each about 50 ms, which is the coarsest granularity available and therefore the one that pays the least overhead per unit of work. This is measured in part 5 rather than assumed.

### Checkpoint — Synthesis 2/4

Dependency structure of one outer iteration:

```
   X, y (read-only)          grid (read-only)
        |                         |
        +----> inner_splits(24)   |
                   |              |
                   v              v
        [config h] x [fold k]  ->  36 x 5 independent fits
                   |
                   v  mean over k            (reduction 1)
             36 mean scores
                   |
                   v  min over h             (reduction 2)
             best configuration
                   |
                   v  refit on all 24, predict observation i
             one output slot
```

| Object | Ownership |
|:--|:--|
| `X`, `y`, grid | shared, read-only |
| train/validation indices | private to each task |
| fitted model, fold error | private to each task |
| external prediction | one slot per outer iteration, one writer |
| best score / parameters | **not** shared; each task returns its own score and one thread reduces afterwards |

The synchronisation points are the two reductions and the join at the end of each pool. There is no lock anywhere, because there is no shared mutable state to protect.

## 4 — Three explicit decompositions

The brief offers three places to cut the work. All three are implemented against the same primitives, so the only difference is where tasks are created.

In [5]:
for fn in (ncv.nested_cv_outer_parallel, ncv.nested_cv_hyper_parallel,
           ncv.nested_cv_inner_parallel):
    print(inspect.getsource(fn))

def nested_cv_outer_parallel(X, y, pool, grid=None, kind="tree"):
    """Decomposition A: one task per outer LOOCV iteration (25 tasks).

    Task i owns output slot i.  Nothing is shared except the read-only data and
    grid, so there is no lock anywhere; the only synchronization is the join
    inside pool.map, which also restores the original order.
    """
    grid = grid if grid is not None else build_grid()
    task = functools.partial(_outer_task, X=X, y=y, grid=grid, kind=kind)
    return list(pool.map(task, range(X.shape[0])))

def nested_cv_hyper_parallel(X, y, pool, grid=None, kind="tree"):
    """Decomposition B: outer loop serial, 36 configuration tasks inside it."""
    grid = grid if grid is not None else build_grid()
    return [outer_iteration_hyper_parallel(i, X, y, grid, pool, kind)
            for i in range(X.shape[0])]

def nested_cv_inner_parallel(X, y, pool, grid=None, kind="tree"):
    """Decomposition C: outer and grid loops serial, 5 fold tasks at the bottom

### Why the reduction is done by the parent

The brief warns against this shape:

```python
if score < shared_best_score:        # a race: the test and the update
    lock()                           # are not one atomic step
    shared_best_score = score
    unlock()
```

Even with the lock, two tasks can both pass the test before either takes it. The version used here avoids the question: each task returns its own score, the parent collects them in grid order, and `select_best` runs once in one thread. No lock, no race, and the result cannot depend on the order in which the tasks happened to finish.

### Correctness

Every decomposition must reproduce the sequential external predictions exactly.

In [6]:
ref = ncv.predictions_of(seq)
rows = []
for label, runner in (('A  outer-fold', ncv.nested_cv_outer_parallel),
                      ('B  hyperparameter', ncv.nested_cv_hyper_parallel),
                      ('C  inner-fold', ncv.nested_cv_inner_parallel)):
    with ThreadPoolExecutor(max_workers=4) as pool:
        got = runner(X, y, pool, grid)
    err = float(np.abs(ncv.predictions_of(got) - ref).max())
    same = all(a['best_params'] == b['best_params'] for a, b in zip(got, seq))
    rows.append((label, err, same))
    print(f'{label:20s} max |difference| = {err:.3g}   same hyperparameters chosen: {same}')
assert all(e == 0.0 and s for _, e, s in rows)
print('\nPASS: all three decompositions reproduce the sequential result exactly.')

A  outer-fold        max |difference| = 0   same hyperparameters chosen: True


B  hyperparameter    max |difference| = 0   same hyperparameters chosen: True


C  inner-fold        max |difference| = 0   same hyperparameters chosen: True

PASS: all three decompositions reproduce the sequential result exactly.


## 5 — Performance

The workload is fixed: same dataset, grid, seed, fold assignment, model and timing method. Only the number of workers changes. $S_p = T_{\mathrm{seq}} / T_p$ and $E_p = S_p / p$, as the brief defines them.

The notebook runs a short version; `bench_lab5.py` runs the full study with interleaved trials and writes `bench_lab5.json`.

### 5.1 A thread pool

In [7]:
t_ref, _ = best_of(lambda: ncv.nested_cv_sequential(X, y, grid))
print(f'sequential reference T_seq = {t_ref:.3f} s\n')
print(f"{'p':>2} {'time (s)':>9} {'S_p':>6} {'E_p':>6} {'cores busy':>11}")
thread_busy = {}
for p in (1, 2, 4, 8):
    with ThreadPoolExecutor(max_workers=p) as pool:
        t, busy = best_of(lambda: ncv.nested_cv_outer_parallel(X, y, pool, grid))
    thread_busy[p] = busy
    print(f'{p:>2} {t:>9.3f} {t_ref/t:>6.2f} {t_ref/t/p:>6.2f} {busy:>11.2f}')

sequential reference T_seq = 1.168 s

 p  time (s)    S_p    E_p  cores busy


 1     1.182   0.99   0.99        1.00


 2     6.349   0.18   0.09        1.04


 4     7.397   0.16   0.04        1.05


 8     8.045   0.15   0.02        1.05


The thread pool is slower than the sequential run, and `cores busy` stays at about 1 however many workers are configured. The workers are taking turns, not running together.

Before blaming the GIL, check that the harness can see parallelism at all: the same kind of thread pool is given a task that is known to release the GIL.

In [8]:
A = [np.random.rand(700, 700) for _ in range(16)]
B = [np.random.rand(700, 700) for _ in range(16)]
t_serial, _ = best_of(lambda: [A[i] @ B[i] for i in range(16)])
print(f'serial reference {t_serial:.3f} s\n')
print(f"{'p':>2} {'time (s)':>9} {'S_p':>6} {'cores busy':>11}")
for p in (1, 2, 4, 8):
    with ThreadPoolExecutor(max_workers=p) as pool:
        t, busy = best_of(lambda: list(pool.map(lambda i: A[i] @ B[i], range(16))))
    print(f'{p:>2} {t:>9.3f} {t_serial/t:>6.2f} {busy:>11.2f}'
          f"   (scikit-learn at the same p: {thread_busy[p]:.2f})")

serial reference 0.203 s

 p  time (s)    S_p  cores busy


 1     0.212   0.96        0.99   (scikit-learn at the same p: 1.00)


 2     0.122   1.67        1.85   (scikit-learn at the same p: 1.04)


 4     0.081   2.51        3.53   (scikit-learn at the same p: 1.05)


 8     0.053   3.81        6.32   (scikit-learn at the same p: 1.05)


Same pool, same machine, same measurement: numpy's matmul reaches several cores busy and a real speedup, while the nested CV stays at one. The pool is not the problem — scikit-learn's `fit` holds the interpreter lock, so threads cannot help here. This is the opposite of Lab 04, where PyTorch released the GIL inside its kernels and an explicit thread pool did run concurrently.

### 5.2 A process pool

Separate processes have separate interpreters and therefore separate locks. The price is that arguments and results are pickled and sent between processes.

In [9]:
print(f"{'p':>2} {'time (s)':>9} {'S_p':>6} {'E_p':>6}")
for p in (1, 2, 4, 8, 16):
    with ProcessPoolExecutor(max_workers=p) as pool:
        t, _ = best_of(lambda: ncv.nested_cv_outer_parallel(X, y, pool, grid))
    print(f'{p:>2} {t:>9.3f} {t_ref/t:>6.2f} {t_ref/t/p:>6.2f}')

 p  time (s)    S_p    E_p


 1     1.175   0.99   0.99


 2     0.689   1.69   0.85


 4     0.403   2.90   0.73


 8     0.275   4.24   0.53


16     0.194   6.02   0.38


### 5.3 Task granularity

The three decompositions do identical work and differ only in how finely it is cut: 25 tasks, 900 tasks or 4500 tasks. Same pool, same four workers.

In [10]:
print(f"{'decomposition':>18} {'tasks':>7} {'time (s)':>9} {'vs sequential':>14}")
for label, runner, ntasks in (('A outer-fold', ncv.nested_cv_outer_parallel, 25),
                              ('B hyperparameter', ncv.nested_cv_hyper_parallel, 25*len(grid)),
                              ('C inner-fold', ncv.nested_cv_inner_parallel, 25*len(grid)*5)):
    with ProcessPoolExecutor(max_workers=4) as pool:
        t, _ = best_of(lambda: runner(X, y, pool, grid))
    print(f'{label:>18} {ntasks:>7} {t:>9.3f} {t_ref/t:>13.2f}x')

     decomposition   tasks  time (s)  vs sequential


      A outer-fold      25     0.393          2.97x


  B hyperparameter     900     0.710          1.64x


      C inner-fold    4500     3.813          0.31x


### Checkpoint — Synthesis 3/4

Decomposition A with a process pool is the version that works: tasks are created by `pool.map` over the 25 outer indices and joined when `map` is drained, with no lock anywhere because the only shared objects are read-only. Two limits are visible in the measurements above, and both are measured rather than asserted:

* **The interpreter lock.** A thread pool gives $S_p < 1$ and `cores busy` $\approx 1$ at every $p$, while the same pool running numpy's matmul reaches several cores busy. Threads are the wrong tool for scikit-learn estimators.
* **Task granularity.** At a fixed four workers, cutting the same work into 4500 tasks instead of 25 is slower than not parallelising at all. One inner fold is about 0.3 ms of work and a process task costs more than that to dispatch.

Efficiency also falls as $p$ grows even for the process pool, because 25 tasks cannot be divided evenly among 16 workers and the last wave leaves most of them idle.

## 6 — The final model, and what the folds chose

Nested CV estimates how the *procedure* generalises. The model to deploy is selected afterwards, by one 5-fold CV over all 25 observations, and then fitted on all 25. Its performance is still the nested-CV figure: there is no observation left to test it on.

In [11]:
final_model, final_params, final_score = ncv.final_model_selection(X, y, grid)
print('final configuration:', final_params, f'(mean 5-fold MAE {final_score:.4f})')

counts = {}
for r in seq:
    key = tuple(sorted(r['best_params'].items()))
    counts[key] = counts.get(key, 0) + 1
print(f'\nthe 25 outer folds selected {len(counts)} different configurations:')
for k, v in sorted(counts.items(), key=lambda kv: -kv[1])[:5]:
    print(f'  {v:2d}x  {dict(k)}')

final configuration: {'max_depth': 2, 'min_samples_split': 2, 'min_samples_leaf': 3} (mean 5-fold MAE 1.6461)

the 25 outer folds selected 10 different configurations:
  10x  {'max_depth': 3, 'min_samples_leaf': 3, 'min_samples_split': 2}
   3x  {'max_depth': 4, 'min_samples_leaf': 3, 'min_samples_split': 2}
   3x  {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2}
   2x  {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 4}
   2x  {'max_depth': 4, 'min_samples_leaf': 2, 'min_samples_split': 2}


## 7 — Extensions: gradient boosting and XGBoost

The same validation protocol, a stronger model. XGBoost is given `n_jobs=1` so that it does not start its own threads inside each task: with a process pool already using every core, a library that also threads internally would oversubscribe the machine and make the speedup curve unreadable.

In [12]:
summary = [('decision tree', ncv.metrics(seq))]
boost_grid = ncv.build_grid(ncv.BOOSTING_GRID)
seq_boost = ncv.nested_cv_sequential(X, y, boost_grid, kind='boosting')
summary.append(('gradient boosting', ncv.metrics(seq_boost)))
try:
    seq_xgb = ncv.nested_cv_sequential(X, y, ncv.build_grid(ncv.XGB_GRID), kind='xgb')
    summary.append(('XGBoost (n_jobs=1)', ncv.metrics(seq_xgb)))
except ImportError as exc:
    print('XGBoost not installed, extension 2 skipped:', exc)

print(f"{'model':>20} {'MAE':>7} {'RMSE':>7} {'R2':>7}")
for name, m in summary:
    print(f"{name:>20} {m['MAE']:>7.3f} {m['RMSE']:>7.3f} {m['R2']:>7.3f}")

               model     MAE    RMSE      R2
       decision tree   1.691   2.151   0.325
   gradient boosting   1.106   1.512   0.667
  XGBoost (n_jobs=1)   1.175   1.546   0.651


### Checkpoint — Synthesis 4/4

Under one protocol, on the 25 external predictions, boosting clearly beats the single tree and XGBoost lands beside it. That is not a general law: a single tree on 25 observations has to cut the smooth response into a handful of constant regions, which boosting's additive ensemble does not. The useful observation is the opposite one — **more capacity does not guarantee better generalisation**, it only helps when the extra capacity matches the shape of the problem, and nested CV is what tells the two cases apart.

The hyperparameter choice is not stable: the 25 outer folds selected 10 different configurations, and the final 5-fold selection over all 25 observations picked yet another depth. With 24 training points, removing one changes which configuration wins. This is exactly why the reported performance must come from the external predictions and not from the final refitted model, and why a single hold-out split on this dataset would be close to meaningless.

**Reproducibility.** Fixed inside every speedup curve: the dataset and seed, the grid, the fold assignment, the model type, the library versions, the timing method and the workload. Recorded separately: CPU model and core count, worker count, the native thread limits, and the host load. A different machine gets its own curve rather than extra points on this one.